<a href="https://colab.research.google.com/github/tristanengst/fintechCS9832025fa/blob/main/4Multi_task_financial_LLMs_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CASE4 Multi-task financial LLMs Models**

**Purpose**:

This notebook demonstrates the multi-task capabilities of FinGPT's financial language models. The supported tasks include:

*   Financial Sentiment Analysis

*   Headline Classification

*   Named Entity Recognition

*   Financial Relation Extraction



---
## Instructions

**Step 1 – Switch to A100 runtime (Colab only)**  
> Purpose: loading FinGPT v3.2 (LLaMA2-based) requires a powerful GPU italicized text, such as A100 or RTX 3090.  

1. Go to **Runtime → Change runtime type** in the top menu.  
2. Set **Hardware accelerator** to `A100 GPU`.  
3. To access A100 in Google Colab, you may need to:  
   - Upgrade to *Colab Pro*, or  
   - Use *Colab Pay-As-You-Go* (**recommended for flexibility**)  
   - [Click here](https://colab.research.google.com/signup) to upgrade or enable Pay-As-You-Go.  
4. Click **Save**.

**Step 2 - Access Requirement for LLaMA-2-7B-Chat-HF Model**  
  This notebook uses the meta-llama/Llama-2-7b-chat-hf model from Hugging Face.  
  To run this code, you must request access from Meta and Hugging Face:
  1. Go to the [model page](https://huggingface.co/meta-llama/Llama-2-7b-chat-hf)
  2. Click “Expand to review and access”
  3. Fill in your contact information as required and submit the request.
  4. Wait for the approval email from HuggingFace. It usually takes around one business day. The email subject will be: [Access granted] Your request to access model meta-llama/Llama-2-70b-hf has been accepted.

Once your request is approved, you can proceed to the next step in this notebook.

**Step 3 – Set your API key**  
 please set your API keys as environment variables to avoid exposing sensitive information in the code.

  On the left sidebar, click the “🔑 Secrets” icon.

  Add the following secret:

1. HF_TOKEN
 - HF_TOKEN refers to your Hugging Face access token.
 - Visit: https://huggingface.co/settings/tokens

Enable "Notebook Access" on the left side of each secret.

**Step 4 – Run the entire notebook**  
Once the dependencies are installed and runtime is set, go to **Runtime → Run all** to execute all cells.

---

## Code

### Install Dependencies and Load FinGPT Code

In [1]:
# You may see an error (e.g. pip resolver warning or "Failed building wheel for tokenizers") from the output.
# It's safe to ignore it for this notebook.
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu126
!pip install transformers==4.32.0 peft==0.5.0
!pip uninstall -y torchao
!git clone https://github.com/AI4Finance-Foundation/FinGPT.git
%cd FinGPT/fingpt/FinGPT_Benchmark/

Looking in indexes: https://download.pytorch.org/whl/cu126
  Using cached transformers-4.32.0-py3-none-any.whl.metadata (118 kB)
  Using cached peft-0.5.0-py3-none-any.whl.metadata (22 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.13.3.tar.gz (314 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using cached transformers-4.32.0-py3-none-any.whl (7.5 MB)
Using cached peft-0.5.0-py3-none-any.whl (85 kB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
error: failed-wheel-build-for-install

× Failed to build ins

In [2]:
# First choose to load data/model from huggingface or local space

FROM_REMOTE = True

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from utils import *

from google.colab import userdata

In [4]:
import logging
# Suppress Warnings during inference
logging.getLogger("transformers").setLevel(logging.ERROR)

### Define Demo Tasks and Inference Functions

In [5]:
demo_tasks = [
    'Financial Sentiment Analysis',
    'Financial Relation Extraction',
    'Financial Headline Classification',
    'Financial Named Entity Recognition',
]
demo_inputs = [
    "Glaxo's ViiV Healthcare Signs China Manufacturing Deal With Desano",
    "Wednesday, July 8, 2015 10:30AM IST (5:00AM GMT) Rimini Street Comment on Oracle Litigation Las Vegas, United States Rimini Street, Inc., the leading independent provider of enterprise software support for SAP AG’s (NYSE:SAP) Business Suite and BusinessObjects software and Oracle Corporation’s (NYSE:ORCL) Siebel , PeopleSoft , JD Edwards , E-Business Suite , Oracle Database , Hyperion and Oracle Retail software, today issued a statement on the Oracle litigation.",
    'april gold down 20 cents to settle at $1,116.10/oz',
    'Subject to the terms and conditions of this Agreement , Bank agrees to lend to Borrower , from time to time prior to the Commitment Termination Date , equipment advances ( each an " Equipment Advance " and collectively the " Equipment Advances ").',
]
demo_instructions = [
    'What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}.',
    'Given phrases that describe the relationship between two words/phrases as options, extract the word/phrase pair and the corresponding lexical relationship between them from the input text. The output format should be "relation1: word1, word2; relation2: word3, word4". Options: product/material produced, manufacturer, distributed by, industry, position held, original broadcaster, owned by, founded by, distribution format, headquarters location, stock exchange, currency, parent organization, chief executive officer, director/manager, owner of, operator, member of, employer, chairperson, platform, subsidiary, legal form, publisher, developer, brand, business division, location of formation, creator.',
    'Does the news headline talk about price in the past? Please choose an answer from {Yes/No}.',
    'Please extract entities and their types from the input sentence, entity types should be chosen from {person/organization/location}.',
]

In [6]:
def load_model(base_model, peft_model, from_remote=False):

    model_name = parse_model_name(base_model, from_remote)

    model = AutoModelForCausalLM.from_pretrained(
        model_name, trust_remote_code=True,
        device_map="auto",
        offload_folder="./offload",
    )
    model.model_parallel = True

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    tokenizer.padding_side = "left"
    if base_model == 'qwen':
        tokenizer.eos_token_id = tokenizer.convert_tokens_to_ids('<|endoftext|>')
        tokenizer.pad_token_id = tokenizer.convert_tokens_to_ids('<|extra_0|>')
    if not tokenizer.pad_token or tokenizer.pad_token_id == tokenizer.eos_token_id:
        tokenizer.add_special_tokens({'pad_token': '[PAD]'})
        model.resize_token_embeddings(len(tokenizer))

    model = PeftModel.from_pretrained(model, peft_model)
    model = model.eval()
    return model, tokenizer


def test_demo(model, tokenizer):

    for task_name, input, instruction in zip(demo_tasks, demo_inputs, demo_instructions):
        prompt = 'Instruction: {instruction}\nInput: {input}\nAnswer: '.format(
            input=input,
            instruction=instruction
        )
        inputs = tokenizer(
            prompt, return_tensors='pt',
            padding=True, max_length=512,
            return_token_type_ids=False
        )
        inputs = {key: value.to(model.device) for key, value in inputs.items()}
        res = model.generate(
            **inputs, max_length=512, do_sample=False,
            eos_token_id=tokenizer.eos_token_id
        )
        output = tokenizer.decode(res[0], skip_special_tokens=True)
        print(f"\n==== {task_name} ====\n")
        print(output)


### Download LLaMA-2 Base Model and Run Demo


In [7]:
# Huggingface login
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
!hf auth login --token $HF_TOKEN
# OS: I'm getting this error from HF: Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-2-7b-hf/resolve/01c7f73d771dfac7d292323805ebc428287df4f9/.gitattributes.
# Access to model meta-llama/Llama-2-7b-hf is restricted and you are not in the authorized list. Visit https://huggingface.co/meta-llama/Llama-2-7b-hf to ask for access.
# Do we need to add instructions to become authorized to the model meta-llama/Llama-2-7b-hf?

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Imagenet` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


There are six ready-to-use financial multi-task language models available from FinGPT, including:

  | Models | Description  | Function |
  | --------- | --------------------- |---------------- |
  | [fingpt-mt_llama2-7b_lora](https://huggingface.co/FinGPT/fingpt-mt_llama2-7b_lora)| Fine-tuned Llama2-7b model with LoRA | Multi-Task |
  | [fingpt-mt_falcon-7b_lora](https://huggingface.co/FinGPT/fingpt-mt_falcon-7b_lora)| Fine-tuned falcon-7b model with LoRA  | Multi-Task |
  | [fingpt-mt_bloom-7b1_lora](https://huggingface.co/FinGPT/fingpt-mt_bloom-7b1_lora) | Fine-tuned bloom-7b1 model with LoRA | Multi-Task |
  | [fingpt-mt_mpt-7b_lora](https://huggingface.co/FinGPT/fingpt-mt_mpt-7b_lora) | Fine-tuned mpt-7b model with LoRA | Multi-Task |
  | [fingpt-mt_chatglm2-6b_lora](https://huggingface.co/FinGPT/fingpt-mt_chatglm2-6b_lora) | Fine-tuned chatglm-6b model with LoRA | Multi-Task |
  | [fingpt-mt_qwen-7b_lora](https://huggingface.co/FinGPT/fingpt-mt_qwen-7b_lora) | Fine-tuned qwen-7b model with LoRA | Multi-Task |

`fingpt-mt_llama2-7b_lora` is used as an example to demonstrate how to perform multiple financial NLP tasks.

In [8]:
!mkdir -p base_models/llama2-7b
!hf download \
    meta-llama/Llama-2-7b-hf \
    --local-dir base_models/llama2-7b \
    --repo-type model \
    # --resume-download

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 17 files: 100% 17/17 [00:00<00:00, 451.62it/s]
Download complete:           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B                         ✓ Downloaded
  path: /content/FinGPT/fingpt/FinGPT_Benchmark/base_models/llama2-7b
Download complete:           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            


In [9]:
base_model = 'llama2'
peft_model = 'FinGPT/fingpt-mt_llama2-7b_lora' if FROM_REMOTE else 'finetuned_models/MT-llama2-linear_202309241345'

model, tokenizer = load_model(base_model, peft_model, FROM_REMOTE)
test_demo(model, tokenizer)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

adapter_model.bin: reconstructing file:   0%|          |  0.00B / 12.6MB            

adapter_model.bin: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2369: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(



==== Financial Sentiment Analysis ====

Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}.
Input: Glaxo's ViiV Healthcare Signs China Manufacturing Deal With Desano
Answer:  positive

==== Financial Relation Extraction ====

Instruction: Given phrases that describe the relationship between two words/phrases as options, extract the word/phrase pair and the corresponding lexical relationship between them from the input text. The output format should be "relation1: word1, word2; relation2: word3, word4". Options: product/material produced, manufacturer, distributed by, industry, position held, original broadcaster, owned by, founded by, distribution format, headquarters location, stock exchange, currency, parent organization, chief executive officer, director/manager, owner of, operator, member of, employer, chairperson, platform, subsidiary, legal form, publisher, developer, brand, business division, location of formation, creator.

In [10]:
!nvidia-smi

Mon Oct 26 15:59:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   60C    P0             71W /   72W |   15384MiB /  23034MiB |     98%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Declaration
The example and instructions used here are adapted from the [`demo.ipynb`](https://github.com/AI4Finance-Foundation/FinGPT/blob/master/fingpt/FinGPT_Benchmark/demo.ipynb) provided in the FinGPT repository.